# Lab 3: Bottom-Up Steel Emissions with Uncertainty


---

## What this lab is

* **Part A: emissions pipeline.** Same tasks, same order, same variable names as Lab 3: operating plants 2020-2030, China utilisation by technology, plant-level production, emission factors, company emissions, projection to 2030.
* **Part B : uncertainty quantification.** You will *interrogate* your own Lab 3 numbers with the tools of the Session 4 lecture: where do the numbers come from, how sure can we be, and what would a risk committee need to know?


## Learning goals

1. Rebuild a bottom-up emissions pipeline from plant-level data (Lab 3).
2. Tell **aleatoric, epistemic and scenario** uncertainty apart.
3. Spot a **data-coverage problem** by checking against official statistics.
4. Give a **range instead of a single number** (bootstrap and simulation) and find which input matters most.
5. Compare **scenarios** and communicate the result with calibrated words.

## Conventions

* Questions are **[QUAL]** (2-4 sentences) or **[QUANT]** (a number from your code plus one sentence).
* Write each answer in the cell right under the question.
* `ttpa` = thousand tonnes per annum. Emissions in Part A are in tCO2; in Part B we switch to **MtCO2**.
* Boxes marked **[OPTIONAL]** can be skipped.
* Run **Kernel -> Restart & Run All** before submitting.

## Submission

Work in **groups of up to 4**. Fill in every member before submitting.

1. Full name: Ines Lebard  Student ID: B00820964
2. Full name: Charlotte Le Bihan  Student ID: B00818310
3. Full name: Camille Tanguy  Student ID: B00821659
4. Full name: Oceane Tanios  Student ID: B00822694

**Group / repo name:** `aidams-lab3-lebard-lebihan-tanguy-tanios`  
**Submitter (one person):**  Oceane Tanios
**Repo URL:**  

### What to submit

- This notebook (`lab3.ipynb`) with all tasks completed and cells run (outputs visible)
- A short `README.md` with how to run the notebook

Create a GitHub repo, push the notebook and README, invite the instructor, and paste the **repo URL** above.

**Send submission info to my email (1 email per group).** Include the GitHub repo URL.

**Done when:** all names are filled in and the notebook contains outputs (no need for the instructor to re-run it).


## Setup (given)

In [61]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

DATA_PATH = Path(os.environ.get("GIST_PATH", "gem-download\Plant-level data - Global Iron and Steel Tracker - September 2026 (V1).xlsx"))
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH.resolve()}")

plant_data = pd.read_excel(DATA_PATH, sheet_name="Plant data")
capacity_data = pd.read_excel(DATA_PATH, sheet_name="Plant capacities and status")
production_data = pd.read_excel(DATA_PATH, sheet_name="Plant production")

years = list(range(2020, 2031))
SEED = 42
print(plant_data.shape, capacity_data.shape, production_data.shape)

(1293, 44) (1848, 16) (1393, 10)


# PART A. Emissions pipeline

## A1. Operating plants 2020-2030

**Goal.** Create `operational_plants`: one row per (plant, year) for the years in `years`, with the plant's `Country/Area`, `Status` and nominal crude steel capacity.
A plant operates in year *t* if `start year <= t <= retired year`. **Assumption (Lab 3):** a missing end date means the plant continues operating.

In [62]:
def to_year(s):
    return pd.to_numeric(s.astype(str).str[:4], errors="coerce")

# retired date is only in the plant sheet so grab it from there
plants = capacity_data.merge(plant_data[["GEM plant ID", "Retired date"]], on="GEM plant ID", how="left")
plants["start_year"] = to_year(plants["Start date"])
plants["end_year"] = to_year(plants["Retired date"]).fillna(9999)  # no end date = still running
plants["Nominal crude steel capacity (ttpa)"] = pd.to_numeric(plants["Nominal crude steel capacity (ttpa)"], errors="coerce")

operational_plants = (
    plants.merge(pd.DataFrame({"Year": years}), how="cross")
    .query("start_year <= Year <= end_year")
    [["GEM plant ID", "Year", "Country/area", "Status", "Main production equipment", "Nominal crude steel capacity (ttpa)"]]
    .reset_index(drop=True)
)

operational_plants.head()

,GEM plant ID,Year,Country/area,Status,Main production equipment,Nominal crude steel capacity (ttpa)
0,P100000120882,2020,Türkiye,operating,EAF,"1,100.00"
1,P100000120882,2021,Türkiye,operating,EAF,"1,100.00"
2,P100000120882,2022,Türkiye,operating,EAF,"1,100.00"
3,P100000120882,2023,Türkiye,operating,EAF,"1,100.00"
4,P100000120882,2024,Türkiye,operating,EAF,"1,100.00"


### Q1 [QUANT] Is every "operating" plant really operating?

Using your `operational_plants` for **China**: for the years **2020 and 2026**, how many plants and how much capacity (ttpa) are counted as operating although their **`Status`** is not an operating status (look at the values of `Status` first)?
Give the share of the Chinese total.

In [63]:
print(operational_plants["Status"].value_counts())

cap = "Nominal crude steel capacity (ttpa)"
china = operational_plants[(operational_plants["Country/area"] == "China") & operational_plants["Year"].isin([2020, 2026])]
# anything that isn't operating / operating pre-retirement
bad = china[~china["Status"].isin(["operating", "operating pre-retirement"])]

q1 = pd.DataFrame({
    "plants": bad.groupby("Year")["GEM plant ID"].nunique(),
    "capacity_ttpa": bad.groupby("Year")[cap].sum(),
})
q1["share_plants"] = q1["plants"] / china.groupby("Year")["GEM plant ID"].nunique()
q1["share_capacity"] = q1["capacity_ttpa"] / china.groupby("Year")[cap].sum()
q1

Status
operating                    9979
retired                      1838
operating pre-retirement     1060
mothballed                    761
cancelled                     489
announced                     357
construction                  264
mothballed pre-retirement      77
Name: count, dtype: int64


,plants,capacity_ttpa,share_plants,share_capacity
Year,,,,
2020,185,"314,997.00",0.45,0.24
2026,131,"192,523.00",0.35,0.15


**Your answer (Q1):**

In 2020, 185 plants (45%) and 314,997 ttpa (24% of capacity) have a non-operating status, vs 131 plants (35%) and 192,523 ttpa (15%) in 2026. So we're overcounting capacity quite a bit, a lot of it from retired plants with no end date.

## A2. China production and capacity

**Goal.** Create
* `crude_steel`: plant-year production (crude steel only), with `Country/Area`, `Owner`, `Parent` and `Main production equipment` attached;
* `china_production`: China total production by year (a Series indexed by year);
* `china_operational`: the rows of `operational_plants` for China;
* `china_capacity`: China total capacity by year.

In [64]:
prod = production_data[production_data["Type of production"] == "Crude steel production (ttpa)"]
crude_steel = prod.melt(id_vars="GEM plant ID", value_vars=[c for c in prod.columns if isinstance(c, int)],
                        var_name="Year", value_name="Production_ttpa")
crude_steel["Production_ttpa"] = pd.to_numeric(crude_steel["Production_ttpa"], errors="coerce")  # 'unknown' and '>0' -> NaN
crude_steel = crude_steel.dropna(subset=["Production_ttpa"]).merge(
    plant_data[["GEM plant ID", "Country/area", "Owner", "Parent (English)", "Main production equipment"]],
    on="GEM plant ID", how="left")

china_production = crude_steel[crude_steel["Country/area"] == "China"].groupby("Year")["Production_ttpa"].sum()
china_operational = operational_plants[operational_plants["Country/area"] == "China"].copy()
china_capacity = china_operational.groupby("Year")[cap].sum()

pd.DataFrame({"production": china_production, "capacity": china_capacity})

,production,capacity
Year,,
2019,"121,655.00",NaN
2020,"179,824.00","1,339,037.00"
2021,"54,817.00","1,336,290.00"
2022,"122,752.00","1,321,524.00"
2023,"97,068.00","1,288,024.00"
2024,"77,204.00","1,279,104.00"
2025,NaN,"1,278,504.00"
2026,NaN,"1,279,207.00"
2027,NaN,"1,278,032.00"


## A3. Utilisation by technology

**Goal.** Compute the utilisation rate (production / capacity) separately for **BF-BOF** and **EAF** (and a residual class "Other").
Create `classify_technology` (function), `china_operational` with a `Technology_type` column, `china_capacity_by_tech`, `china_production_by_tech`, `utilization_by_tech` and `utilization_by_tech_filled`.

In [65]:
def classify_technology(equipment):
    equipment = str(equipment)
    if "BOF" in equipment:
        return "BF-BOF"
    if "EAF" in equipment:
        return "EAF"
    return "Other"

china_operational["Technology_type"] = china_operational["Main production equipment"].map(classify_technology)
crude_steel["Technology_type"] = crude_steel["Main production equipment"].map(classify_technology)

china_capacity_by_tech = china_operational.groupby(["Year", "Technology_type"])[cap].sum().unstack()
china_production_by_tech = (crude_steel[crude_steel["Country/area"] == "China"]
                            .groupby(["Year", "Technology_type"])["Production_ttpa"].sum().unstack())
utilization_by_tech = (china_production_by_tech / china_capacity_by_tech).dropna(how="all")

# carry the last known rate forward, "Other" has no production so just use the overall rate
overall = (china_production / china_capacity).reindex(years).ffill()
utilization_by_tech_filled = utilization_by_tech.reindex(years).ffill().apply(lambda col: col.fillna(overall))

utilization_by_tech_filled

Technology_type,BF-BOF,EAF,Other
Year,,,
2020,0.15,0.02,0.13
2021,0.05,0.01,0.04
2022,0.10,0.04,0.09
2023,0.08,0.04,0.08
2024,0.07,0.01,0.06
2025,0.07,0.01,0.06
2026,0.07,0.01,0.06
2027,0.07,0.01,0.06
2028,0.07,0.01,0.06


## A4. Plant-level production

**Goal.** Attribute each technology's utilisation rate to its plants: add `Utilization_rate_tech` and `Estimated_production_ttpa` (= capacity x utilisation) to `china_operational`.

In [66]:
rates = utilization_by_tech_filled.stack().rename("Utilization_rate_tech").reset_index()
china_operational = china_operational.merge(rates, on=["Year", "Technology_type"], how="left")
china_operational["Estimated_production_ttpa"] = china_operational[cap] * china_operational["Utilization_rate_tech"]

china_operational.groupby("Year")["Estimated_production_ttpa"].sum()

Year
2020   179,856.23
2021    54,826.85
2022   122,774.29
2023    97,086.09
2024    77,290.92
2025    77,144.72
2026    77,009.21
2027    76,864.96
2028    76,864.96
2029    76,864.96
2030    76,864.96
Name: Estimated_production_ttpa, dtype: float64

## A5. Emission factors and plant emissions

**Goal.** Create `emission_factors` (dict), `map_emission_factor` (function), and add `Emission_factor_tCO2_per_t` and `Estimated_emissions_tCO2` to `china_operational`.

Use these emissions factors (tCO2 per tonne crude steel, after Hasanbeigi 2019), keyed by the **exact** equipment string, with **1.6** for anything else:

| `BF; BOF` 2.1 | `BF; BOF; EAF` 1.9 | `BF; BOF; EAF; DRI` 1.9 | `BF; BOF; DRI` 2.0 | `BF; EAF` 1.9 |
|---|---|---|---|---|
| `BF` 2.2 | `EAF` 0.4 | `EAF; DRI` 1.2 | `DRI` 1.3 | `Steel other/unspecified` 1.6 |

> **Hint.** Emissions (t) = production (ttpa) x 1,000 x factor. `Series.map` with a function, or `dict.get(key, 1.6)`.

In [67]:
emission_factors = {
    "BF; BOF": 2.1, "BF; BOF; EAF": 1.9, "BF; BOF; EAF; DRI": 1.9, "BF; BOF; DRI": 2.0, "BF; EAF": 1.9,
    "BF": 2.2, "EAF": 0.4, "EAF; DRI": 1.2, "DRI": 1.3, "Steel other/unspecified": 1.6,
}

def map_emission_factor(equipment):
    return emission_factors.get(equipment, 1.6)

china_operational["Emission_factor_tCO2_per_t"] = china_operational["Main production equipment"].map(map_emission_factor)
china_operational["Estimated_emissions_tCO2"] = (china_operational["Estimated_production_ttpa"] * 1000
                                                 * china_operational["Emission_factor_tCO2_per_t"])

china_operational.groupby("Year")["Estimated_emissions_tCO2"].sum()

Year
2020   359,129,054.16
2021   108,312,508.51
2022   238,866,407.99
2023   186,306,056.59
2024   152,187,805.61
2025   151,865,819.41
2026   151,492,947.85
2027   151,158,844.34
2028   151,158,844.34
2029   151,158,844.34
2030   151,158,844.34
Name: Estimated_emissions_tCO2, dtype: float64

## A6. Company emissions

**Goal.** Create `Company` (= `Parent`, or `Owner` when `Parent` is missing), `company_emissions` (columns `Year`, `Company`, `estimated_emissions_tCO2`) and `top_companies_latest` (top 10 in the **latest year**).

In [68]:
china_operational = china_operational.merge(plant_data[["GEM plant ID", "Owner", "Parent (English)"]], on="GEM plant ID", how="left")
china_operational["Company"] = china_operational["Parent (English)"].fillna(china_operational["Owner"])

company_emissions = (china_operational.groupby(["Year", "Company"])["Estimated_emissions_tCO2"].sum()
                     .rename("estimated_emissions_tCO2").reset_index())

top_companies_latest = (company_emissions[company_emissions["Year"] == company_emissions["Year"].max()]
                        .nlargest(10, "estimated_emissions_tCO2"))
top_companies_latest

,Year,Company,estimated_emissions_tCO2
3497,2030,Angang Steel Co Ltd [100.0%],"4,677,512.94"
3509,2030,Baoshan Iron & Steel Co Ltd [100.0%],"4,409,219.01"
3522,2030,Beijing Shougang Co Ltd [100.0%],"3,069,897.65"
3688,2030,Jingye Group Co Ltd [100.0%],"2,505,208.05"
3667,2030,Inner Mongolia BaoTou Steel Union Co Ltd [100.0%],"2,486,815.05"
3675,2030,Jiangsu Shagang Iron & Steel Co Ltd [100.0%],"2,481,626.22"
3529,2030,British Asia Pan Ruogen International Co Ltd [...,"2,415,207.67"
3715,2030,Maanshan Iron & Steel Co Ltd [100.0%],"2,395,764.61"
3663,2030,Hunan Valin Steel Co Ltd [100.0%],"2,311,827.49"
3571,2030,Fujian Metallurgical Holding Co Ltd [53.4%]; u...,"2,290,345.28"


### Q2 [QUAL] Look at the company names

Read the names in `top_companies_latest`. Give **two data-quality problems** you notice, and explain how each could bias a *financed-emissions* figure for a lender (hint: ownership shares, joint ventures).

**Your answer (Q2):**

Some names have ownership shares mixed in, like "[59.2%]; unknown [40.8%]", but we still give the whole plant's emissions to them, so a lender to a partial owner would be charged too much. Also, the same group shows up under different names (Baoshan and Maanshan are both Baowu), so the group's total gets split up and its emissions look smaller than they are.

## A7. Projection to 2030

**Goal.** Create `actual_years` (years with production data), `forecast_years` (the years after the last observed one), `projected_company_emissions` (company emissions in the forecast years) and a function `top_company_emissions(df, year)` returning the top 5 companies of that year. Print the top 5 for the first three forecast years.

In [69]:
actual_years = sorted(set(china_production.index) & set(years))
forecast_years = [y for y in years if y > max(actual_years)]
projected_company_emissions = company_emissions[company_emissions["Year"].isin(forecast_years)]

def top_company_emissions(df, year):
    return df[df["Year"] == year].nlargest(5, "estimated_emissions_tCO2")

print(actual_years, forecast_years)
for y in forecast_years[:3]:
    print(f"\n{y}")
    print(top_company_emissions(projected_company_emissions, y))

[2020, 2021, 2022, 2023, 2024] [2025, 2026, 2027, 2028, 2029, 2030]

2025
      Year                                            Company  \
1796  2025                       Angang Steel Co Ltd [100.0%]   
1808  2025               Baoshan Iron & Steel Co Ltd [100.0%]   
1820  2025                   Beijing Shougang Co Ltd [100.0%]   
1988  2025                       Jingye Group Co Ltd [100.0%]   
1966  2025  Inner Mongolia BaoTou Steel Union Co Ltd [100.0%]   

      estimated_emissions_tCO2  
1796              4,677,512.94  
1808              4,409,219.01  
1820              3,062,112.90  
1988              2,505,208.05  
1966              2,486,815.05  

2026
      Year                                            Company  \
2140  2026                       Angang Steel Co Ltd [100.0%]   
2152  2026               Baoshan Iron & Steel Co Ltd [100.0%]   
2165  2026                   Beijing Shougang Co Ltd [100.0%]   
2331  2026                       Jingye Group Co Ltd [100.0%]   
2310  

### Q3 [QUAL] Forecast, projection or scenario?

The top-5 tables for the first projected years are (almost) identical. (a) Why? (b) Is this a **forecast**, a **projection** or a **scenario**, and what is it conditional on? (c) Name **three things** a fan chart would tell a risk manager that this table does not.

**Your answer (Q3):**

They're basically the same because we just copy the 2024 utilisation forward and capacity barely changes. So it's a projection, it only holds if 2024 utilisation and today's plants stay the same until 2030. A fan chart would show how uncertain the numbers are, how that grows over time, and if the ranking could change.

---
# PART B. How much can we trust these numbers?

You now have company emissions as **single numbers**. Part B asks three simple questions:

1. **Do the numbers agree with an outside reference?** (reality check)
2. **How wide is the range of plausible values?** (an interval, found by simulation)
3. **What does the range depend on?** (inputs and scenarios)

**Three words you need**

| Word | Meaning | Example |
|---|---|---|
| **Aleatoric** uncertainty | pure randomness, more data does *not* remove it | will a given year be a strong or a weak year for steel? |
| **Epistemic** uncertainty | we lack knowledge, more data or better models *can* reduce it | what is the true average emission factor of a blast furnace? |
| **Scenario** uncertainty | depends on future choices, it is not a probability | will China retire old blast furnaces by 2030? |

A **90% interval** is a range that contains the true value in 90% of cases *if our assumptions are right*.

## B0. Think first

> **Pause & think.** No code.

### Q4 [QUAL] Which kind of uncertainty?

For each input of your pipeline, write **A** (aleatoric), **E** (epistemic) or **S** (scenario), and say whether **more data could reduce it** (yes / no). One short line each.

| Input | A / E / S | More data helps? |
|---|---|---|
| Utilisation rate of a blast furnace next year | | |
| Emission factor of BF-BOF steel | | |
| Share of plants that report their production | | |
| Steel demand in 2030 | | |

**Your answer (Q4):**

| Input | A / E / S | More data helps? |
|---|---|---|
| Utilisation rate of a blast furnace next year | A | no |
| Emission factor of BF-BOF steel | E | yes |
| Share of plants that report their production | E | yes |
| Steel demand in 2030 | S | no |

## B1. Reality check against official statistics

Official crude steel production of China (**rounded worldsteel figures: verify them and cite your source**):

| Year | 2020 | 2021 | 2022 | 2023 |
|---|---|---|---|---|
| Mt | 1,065 | 1,035 | 1,015 | 1,020 |

**Goal.** Create `official_kt` (dict, in **ttpa**: 1 Mt = 1,000 ttpa) and a DataFrame `check` with, for 2020-2023: the production in your dataset (`china_production`), the official value, the **coverage** (dataset / official) and the **implied utilisation** (official / `china_capacity`).

> **Hint.** `pd.Series(official_kt)`, `pd.DataFrame({...})`, `.dropna()`.

In [70]:
# source: worldsteel, World Steel in Figures
official_kt = {2020: 1_065_000, 2021: 1_035_000, 2022: 1_015_000, 2023: 1_020_000}

check = pd.DataFrame({
    "china_production": china_production,
    "official": pd.Series(official_kt),
    "china_capacity": china_capacity,
}).dropna()
check["coverage"] = check["china_production"] / check["official"]
check["implied_utilisation"] = check["official"] / check["china_capacity"]
check["underestimate_factor"] = check["official"] / check["china_production"]
check

,china_production,official,china_capacity,coverage,implied_utilisation,underestimate_factor
2020,"179,824.00","1,065,000.00","1,339,037.00",0.17,0.80,5.92
2021,"54,817.00","1,035,000.00","1,336,290.00",0.05,0.77,18.88
2022,"122,752.00","1,015,000.00","1,321,524.00",0.12,0.77,8.27
2023,"97,068.00","1,020,000.00","1,288,024.00",0.10,0.79,10.51


### Q5 [QUANT + QUAL] What does your pipeline miss?

(a) Give the **coverage** and **implied utilisation** for 2023, and the factor by which your Part A pipeline **under-estimates** China's production. (b) The BF-BOF "utilisation" in Part A drops from about 57% to about 8% between 2020 and 2021. Is that a real change in how furnaces are run, or something else? Which kind of uncertainty is it (A, E or S)?

**Your answer (Q5):**

(a) In 2023 coverage is 0.10 and implied utilisation is 0.79, so our pipeline underestimates China's production by a factor of 10.51. (b) In our data BF-BOF utilisation goes from 0.15 in 2020 to 0.05 in 2021, but that's not furnaces running less, it's just fewer plants reporting production while we still divide by all the capacity. That's epistemic uncertainty (E), better data would fix it.

## B2. Fix the denominator: compare like with like

Use only plants that **report production** *and* have a capacity ("matched" plants), so that numerator and denominator describe the same plants.

**Goal.** Create `obs_matched` (one row per plant-year with production **and** capacity > 0, with `Technology_type` and `u` = production / capacity) and `utilization_matched` (year x technology: **sum of production / sum of capacity** of the matched plants).

> **Hint.** Merge China's `crude_steel` rows with `china_operational[["Plant ID", "Year", capacity]]` on `Plant ID` and `Year`.

In [71]:
# some plants have 2 production rows so sum them, same for capacity (several units per plant)
china_cs = (crude_steel[crude_steel["Country/area"] == "China"]
            .groupby(["GEM plant ID", "Year", "Technology_type"], as_index=False)["Production_ttpa"].sum())
plant_cap = china_operational.groupby(["GEM plant ID", "Year"], as_index=False)[cap].sum()

obs_matched = china_cs.merge(plant_cap, on=["GEM plant ID", "Year"])
obs_matched = obs_matched[(obs_matched["Production_ttpa"] > 0) & (obs_matched[cap] > 0)].copy()
obs_matched["u"] = obs_matched["Production_ttpa"] / obs_matched[cap]

sums = obs_matched.groupby(["Year", "Technology_type"])[["Production_ttpa", cap]].sum()
utilization_matched = (sums["Production_ttpa"] / sums[cap]).unstack()
utilization_matched

Technology_type,BF-BOF,EAF
Year,,
2020,0.84,0.92
2021,0.75,0.64
2022,0.86,0.86
2023,0.79,0.69
2024,0.83,0.75


### Q6 [QUANT + QUAL] Naive or matched?

(a) For **BF-BOF in 2023**, report the Part A (naive) utilisation, the matched utilisation, and the official-implied utilisation (from `check`). (b) Which would you use for projections? In one sentence, say what bias could **remain** in the matched estimate (hint: which plants report their production?).

**Your answer (Q6):**

(a) For BF-BOF in 2023, naive utilisation is 0.08, matched is 0.79 and official-implied is 0.79. (b) We'd use the matched one since it's way more realistic. But the plants that report their production are probably the bigger, better-run ones, so it could still be a bit too high for the whole sector.

## B3. A range from a sample: the bootstrap

We have a matched utilisation for BF-BOF, but it comes from a **sample** of plants. A different sample would give a slightly different value. How different? The **bootstrap** answers it in four steps:

1. Draw plants **with replacement** from your sample, as many as the sample contains.
2. Recompute the average utilisation (sum of production / sum of capacity).
3. Repeat **1,000 times**.
4. The 5th and 95th percentiles of the 1,000 values form a **90% interval**.

**Goal.** Write a function `boot_mean_u(df, n=1000)` that returns the `n` bootstrap values. Use it to create `ci_all` (interval for **all** matched BF-BOF plants) and `ci_small` (interval for **10 randomly chosen** BF-BOF plants).

> **Hint.** Sum production and capacity **per plant** first (`groupby("Plant ID")`), then resample plant numbers with `np.random.default_rng(42).integers(0, n_plants, size=(n, n_plants))`; the values of one draw are `prod[idx].sum(1) / cap[idx].sum(1)`. `np.percentile(values, [5, 95])`.

In [72]:
def boot_mean_u(df, n=1000):
    per_plant = df.groupby("GEM plant ID")[["Production_ttpa", cap]].sum()
    prod, capa = per_plant["Production_ttpa"].values, per_plant[cap].values
    idx = np.random.default_rng(SEED).integers(0, len(per_plant), size=(n, len(per_plant)))
    return prod[idx].sum(1) / capa[idx].sum(1)

bf = obs_matched[obs_matched["Technology_type"] == "BF-BOF"]
ci_all = np.percentile(boot_mean_u(bf), [5, 95])

small_ids = np.random.default_rng(SEED).choice(bf["GEM plant ID"].unique(), 10, replace=False)
ci_small = np.percentile(boot_mean_u(bf[bf["GEM plant ID"].isin(small_ids)]), [5, 95])

print(f"ci_all:   {ci_all.round(2)}  width {ci_all[1] - ci_all[0]:.2f}")
print(f"ci_small: {ci_small.round(2)}  width {ci_small[1] - ci_small[0]:.2f}")

ci_all:   [0.79 0.86]  width 0.07
ci_small: [0.81 0.97]  width 0.16


### Q7 [QUANT + QUAL] Small sample, wide interval

(a) Report `ci_all` and `ci_small` and their widths. (b) Why is the second wider? (c) In one sentence, what does "90% interval" mean here? (d) Which kind of uncertainty (A or E) shrinks when you have more plants?

**Your answer (Q7):**

(a) ci_all is [0.79, 0.86] with a width of 0.07, and ci_small is [0.81, 0.97] with a width of 0.16. (b) With only 10 plants, one weird plant changes the average a lot, so the resampled values move around more. (c) If our assumptions hold, the true average BF-BOF utilisation is in that range 90% of the time. (d) It's epistemic uncertainty that shrinks with more plants.

> ### [OPTIONAL] Bayesian version of the same interval
>
> A **credible interval** answers a more direct question: *given the data and what I believed before, where is the true average?* We start from a **prior guess** (mean $\mu_0$, worth $\kappa_0$ observations) and update it with the data (mean $\bar x$, $n$ observations):
>
> posterior mean $=\dfrac{\kappa_0\,\mu_0 + n\,\bar x}{\kappa_0+n}$ (a weighted average: the prior counts as $\kappa_0$ observations).
>
> **Task.** For the **10 plants** of `ci_small`, compute the posterior mean for (i) a **weak** prior ($\mu_0=0.5$, $\kappa_0=0.01$) and (ii) a **wrong** prior ($\mu_0=0.3$, $\kappa_0=5$). The width of a 90% interval is roughly $1.8\times s/\sqrt{n+\kappa_0}$ ($s$ = standard deviation of the 10 values).
> **Question Q7b.** How much does the wrong prior move the answer, and why does this matter when data are scarce?

In [73]:
per_plant = bf[bf["GEM plant ID"].isin(small_ids)].groupby("GEM plant ID")[["Production_ttpa", cap]].sum()
x = per_plant["Production_ttpa"] / per_plant[cap]
n, xbar, s = len(x), x.mean(), x.std()
print(f"n = {n}, mean = {xbar:.2f}, sd = {s:.2f}\n")

priors = {"weak": (0.5, 0.01), "wrong": (0.3, 5)}
pd.DataFrame({
    name: {"posterior_mean": (k0 * mu0 + n * xbar) / (k0 + n), "width_90": 1.8 * s / np.sqrt(n + k0)}
    for name, (mu0, k0) in priors.items()
}).T

n = 10, mean = 0.92, sd = 0.23



,posterior_mean,width_90
weak,0.92,0.13
wrong,0.71,0.11


**Your answer (Q7b (optional)):**

With the weak prior the posterior mean is 0.92 (width 0.13), basically the same as the data, but the wrong prior pulls it down to 0.71 (width 0.11). That's a big shift from just 10 plants, and the interval even gets narrower, so we'd feel more sure about a worse answer. When data are scarce the prior weighs a lot, so a bad guess can easily take over.

## B4. Simulation: a range for emissions

**Idea (Monte Carlo).** Instead of one value per input, use a *plausible range*, then **repeat the whole calculation many times**, each time with different plausible values. The cloud of results shows the uncertainty of the output.

Two uncertain inputs:

| Input | Plausible range | How to draw |
|---|---|---|
| **Utilisation** per technology | the bootstrap values of B3 | one bootstrap value per draw and technology (all technologies pooled if a technology has fewer than 5 plants) |
| **Emission factor** per technology | multiplier on your Part A factor, **assumed illustrative values** below | triangular distribution (low, mode = 1, high): `rng.triangular(low, 1, high, size=n)` |

| Technology | low | mode | high |
|---|---|---|---|
| BF-BOF | 0.85 | 1 | 1.20 |
| EAF | 0.65 | 1 | 2.00 |
| Other | 0.65 | 1 | 1.40 |

**Goal.** Create `ef_mult_ranges` (dict) and a function `simulate(n=1000, year=2030, vary_util=True, vary_ef=True, bf_cap_mult=1.0, util_mult=1.0, seed=0)` returning a dict with
* `national`: array of `n` national emissions (**MtCO2**) for `year`;
* `national_prod`: array of `n` national productions (**Mt**);
* `by_company`: DataFrame (`n` rows, one column per company, MtCO2).

> **Hint.**
> 1. Take the rows of `china_operational` for `year`. For every plant you need: capacity (missing = 0), technology, your Part A emission factor, company.
> 2. Draw a matrix `u` (n x 3) of utilisations and a matrix `mult` (n x 3) of multipliers (columns = BF-BOF, EAF, Other). If `vary_util=False`, use the pooled matched utilisation of each technology instead (sum of production / sum of capacity over all years); if `vary_ef=False`, use multipliers equal to 1.
> 3. Choose each plant's column by its technology: `u[:, tech_index_of_plant]` gives an (n x plants) matrix. Then production = capacity x utilisation (x `util_mult`), emissions = production x 1,000 x factor x multiplier / 1e6.
> 4. `bf_cap_mult` multiplies the capacity of BF-BOF plants (used for scenarios later).
> 5. Company totals: `pd.DataFrame(em_matrix, columns=company_of_each_plant).T.groupby(level=0).sum().T`.

In [74]:
techs = ["BF-BOF", "EAF", "Other"]
ef_mult_ranges = {"BF-BOF": (0.85, 1.20), "EAF": (0.65, 2.00), "Other": (0.65, 1.40)}

# "Other" has no matched plants so it falls back to everything
tech_obs = {}
for t in techs:
    d = obs_matched[obs_matched["Technology_type"] == t]
    tech_obs[t] = d if d["GEM plant ID"].nunique() >= 5 else obs_matched

boot_u = {t: boot_mean_u(d) for t, d in tech_obs.items()}
point_u = {t: d["Production_ttpa"].sum() / d[cap].sum() for t, d in tech_obs.items()}

def simulate(n=1000, year=2030, vary_util=True, vary_ef=True, bf_cap_mult=1.0, util_mult=1.0, seed=0):
    rng = np.random.default_rng(seed)
    df = china_operational[china_operational["Year"] == year]
    capa = df[cap].fillna(0).values * np.where(df["Technology_type"] == "BF-BOF", bf_cap_mult, 1.0)
    t_idx = df["Technology_type"].map(techs.index).values

    if vary_util:
        u = np.column_stack([rng.choice(boot_u[t], n) for t in techs])
    else:
        u = np.tile([point_u[t] for t in techs], (n, 1))
    if vary_ef:
        mult = np.column_stack([rng.triangular(ef_mult_ranges[t][0], 1, ef_mult_ranges[t][1], n) for t in techs])
    else:
        mult = np.ones((n, len(techs)))

    prod = capa * u[:, t_idx] * util_mult  # ttpa, n x plants
    em = prod * 1000 * df["Emission_factor_tCO2_per_t"].values * mult[:, t_idx] / 1e6  # MtCO2
    return {
        "national": em.sum(1),
        "national_prod": prod.sum(1) / 1000,
        "by_company": pd.DataFrame(em, columns=df["Company"].values).T.groupby(level=0).sum().T,
    }

def summary(x):
    lo, med, hi = np.percentile(x, [5, 50, 95])
    return {"median": med, "low": lo, "high": hi, "rel_half_width": (hi - lo) / (2 * med)}

### Q8 [QUANT + QUAL] Read the result

(a) For **2030**, report the **median** and the **90% interval** (5th to 95th percentile) of national emissions, and the **relative half-width** = (upper - lower) / (2 x median). (b) Compare the median with your **Part A value for 2030** (sum of `company_emissions` for 2030, in Mt). Which one is closer to the simulation, and why do they differ so much (hint: B1)?

In [75]:
res = simulate()
print(pd.Series(summary(res["national"])))

part_a_2030 = company_emissions.loc[company_emissions["Year"] == 2030, "estimated_emissions_tCO2"].sum() / 1e6
print(f"\nPart A 2030: {part_a_2030:,.2f} MtCO2")

median           1,921.12
low              1,719.31
high             2,182.37
rel_half_width       0.12
dtype: float64

Part A 2030: 151.16 MtCO2


**Your answer (Q8):**

(a) For 2030 the median is 1,921.12 MtCO2, with a 90% interval of 1,719.31 to 2,182.37 and a relative half-width of 0.12. (b) Part A only gives 151.16 MtCO2, which is way below the simulation. That's because Part A uses the naive utilisation (around 0.07), which only covers about 10% of real production (B1), while the simulation uses the matched utilisation (around 0.8), so the simulation is the more realistic one.

### Q9 [QUANT + QUAL] Which input matters most?

Run `simulate` twice: once varying **only the utilisation** (`vary_ef=False`) and once varying **only the emission factor** (`vary_util=False`). For each, give the relative half-width for 2030. Which input creates more of the uncertainty? Could **more data** reduce it?

In [76]:
pd.DataFrame({
    "util only": summary(simulate(vary_ef=False)["national"]),
    "EF only": summary(simulate(vary_util=False)["national"]),
})

,util only,EF only
median,"1,896.63","1,943.72"
low,"1,819.70","1,735.12"
high,"1,965.69","2,165.61"
rel_half_width,0.04,0.11


**Your answer (Q9):**

Varying only utilisation gives a relative half-width of 0.04, and varying only the emission factor gives 0.11, so the emission factor drives most of the uncertainty. More data could reduce it, since it's epistemic: measuring the real emissions of plants would narrow the range.

### Q10 [QUANT + QUAL] Companies: can you say who is number one?

Using `by_company` of your 2030 run: (a) for the **top 3 companies by median**, give the median and the 90% interval; (b) compute the probability that the #1 company emits **more than** the #2 (fraction of simulations); (c) what would you write in a report instead of "Company X is the largest emitter"?

In [77]:
by_company = res["by_company"]
top3 = by_company.median().nlargest(3).index

print(pd.DataFrame({c: summary(by_company[c]) for c in top3}).T)
print(f"\nP(#1 > #2): {(by_company[top3[0]] > by_company[top3[1]]).mean():.2f}")

                                      median   low  high  rel_half_width
Angang Steel Co Ltd [100.0%]           57.26 50.75 65.09            0.13
Baoshan Iron & Steel Co Ltd [100.0%]   53.97 47.84 61.36            0.13
Beijing Shougang Co Ltd [100.0%]       37.95 33.75 43.07            0.12

P(#1 > #2): 1.00


**Your answer (Q10):**

(a) Angang has a median of 57.26 MtCO2 [50.75, 65.09], Baoshan 53.97 [47.84, 61.36] and Beijing Shougang 37.95 [33.75, 43.07]. (b) P(#1 > #2) is 1.00, but only because both are fully BF-BOF and every BF-BOF plant gets the same draw in our model, so their ranking can't flip. (c) We'd write something like "Angang and Baoshan are likely the two biggest emitters, with similar ranges", rather than naming one clear number one.

## B5. Is the interval honest? Test it

A 90% interval should contain the true value in about 90% of cases. We have one case where the truth is known: **official 2023 production**.

### Q11 [QUANT + QUAL] Does the official value fall inside?

Run `simulate(1000, year=2023)` and look at `national_prod` (Mt). (a) Give its 90% interval and say whether the official 2023 value (1,020 Mt) falls inside. (b) If not, what does this tell you about the interval, **even though it was computed correctly**? (c) What would you do about it?

In [78]:
res_2023 = simulate(1000, year=2023)
print(pd.Series(summary(res_2023["national_prod"])))
print(f"\nofficial 2023 inside? {summary(res_2023['national_prod'])['low'] <= 1020 <= summary(res_2023['national_prod'])['high']}")

median           1,049.29
low              1,002.91
high             1,085.33
rel_half_width       0.04
dtype: float64

official 2023 inside? True


**Your answer (Q11):**

(a) The 90% interval for 2023 production is 1,002.91 to 1,085.33 Mt, so the official 1,020 Mt is inside. (b) So no problem here, but it's only one check on one year, and the interval only covers what we let vary (utilisation and emission factor), not the capacity data itself, like the retired plants we saw in Q1. (c) We'd test it on other years like 2020-2022 too, and widen it with capacity uncertainty if it keeps missing.

## B6. Scenarios

A **scenario** is a "what if" story. We do **not** attach a probability to it; we run each one through the same simulation and compare.

| Scenario | Story | How |
|---|---|---|
| **S0** | no change | as is |
| **S1** | **30% of BF-BOF capacity is retired** (by 2027, still true in 2030) | `bf_cap_mult=0.7` |
| **S2** | **steel demand drops**: utilisation 10% lower | `util_mult=0.9` |

### Q12 [QUANT + QUAL] Scenario gap vs interval width

(a) Run the three scenarios for 2030 and give the **median and 90% interval** of each. (b) Compare the **gap between** scenario medians with the **width of the interval** within a scenario. Which is bigger? (c) Why would it be misleading to average the three scenarios into one probability-weighted number?

In [79]:
scenarios = {
    "S0": simulate(),
    "S1": simulate(bf_cap_mult=0.7),
    "S2": simulate(util_mult=0.9),
}
scen_table = pd.DataFrame({k: summary(v["national"]) for k, v in scenarios.items()}).T
scen_table["width"] = scen_table["high"] - scen_table["low"]
scen_table["gap_vs_S0"] = scen_table.loc["S0", "median"] - scen_table["median"]
scen_table

,median,low,high,rel_half_width,width,gap_vs_S0
S0,"1,921.12","1,719.31","2,182.37",0.12,463.06,0.00
S1,"1,373.24","1,228.23","1,557.60",0.12,329.38,547.88
S2,"1,729.01","1,547.38","1,964.13",0.12,416.76,192.11


**Your answer (Q12):**

(a) S0 has a median of 1,921.12 MtCO2 [1,719.31, 2,182.37], S1 1,373.24 [1,228.23, 1,557.60] and S2 1,729.01 [1,547.38, 1,964.13]. (b) The gap between S0 and S1 (547.88) is bigger than the width inside S0 (463.06), while the gap with S2 (192.11) is smaller, so retiring blast furnaces matters more than our uncertainty, but a demand drop doesn't. (c) Scenarios are choices, not random outcomes, so there's no real probability to weight them with, and averaging would give a number that matches none of the futures.

## B7. Communicate

Words used by the IPCC for probabilities: **virtually certain** 99-100%, **very likely** 90-100%, **likely** 66-100%, **about as likely as not** 33-66%, **unlikely** 10-33%, **very unlikely** 1-10%, **exceptionally unlikely** 0-1%.

### Q13 [QUANT + QUAL] A probability, a word, and a note

(a) For each scenario, compute the probability that **2030 emissions exceed the 2023 median** (from your `simulate(..., year=2023)`) and give the matching word. (b) Write a note of **about 80 words** for a bank's risk committee about China's steel emissions in 2030. Include: the **median and 90% interval** (baseline), **one sentence on the scenarios**, the **biggest source of uncertainty**, **one limitation** of the model and **one recommended action**. Plain language, no jargon.

> **Hint.** `(draws > threshold).mean()`.

In [80]:
def ipcc_word(p):
    if p >= 0.99: return "virtually certain"
    if p >= 0.90: return "very likely"
    if p >= 0.66: return "likely"
    if p >= 0.33: return "about as likely as not"
    if p >= 0.10: return "unlikely"
    if p >= 0.01: return "very unlikely"
    return "exceptionally unlikely"

threshold = np.median(res_2023["national"])
print(f"2023 median: {threshold:,.2f} MtCO2\n")

probs = pd.DataFrame({"P(2030 > 2023 median)": {k: (v["national"] > threshold).mean() for k, v in scenarios.items()}})
probs["word"] = probs.iloc[:, 0].map(ipcc_word)
probs

2023 median: 1,942.79 MtCO2



,P(2030 > 2023 median),word
S0,0.44,about as likely as not
S1,0.00,exceptionally unlikely
S2,0.07,very unlikely


**Your answer (Q13):**

(a) The 2023 median is 1,942.79 MtCO2. The chance of going above it in 2030 is 0.44 for S0 (about as likely as not), 0.00 for S1 (exceptionally unlikely) and 0.07 for S2 (very unlikely).

(b) Our baseline puts China's steel emissions in 2030 at about 1,921 MtCO2, with a 90% range of 1,719 to 2,182. If 30% of blast furnace capacity is retired, that drops to around 1,373, while a 10% fall in demand only brings it to about 1,729. The biggest uncertainty is the emission factor of each plant. The model relies on incomplete plant data, so company figures are rough. We'd recommend asking big borrowers for their actual plant-level emissions.